# 05 — Application Domain (Williams Plot) ve SHAP Açıklanabilirlik

**Hedef:** Notebook 04'te kaydedilen final modelin hangi kimyasal uzayda güvenilir olduğunu leverage tabanlı application domain ile incelemek ve tahminleri SHAP ile açıklamak.

### Bu notebookta öğreneceklerimiz
- Application domain kavramı
- PCA uzayında leverage hesabı
- Standartlaştırılmış artıklar ve Williams plot
- AD-in / AD-out sınıflaması
- Global ve lokal SHAP açıklamaları
- En önemli descriptorların CSV ve PNG olarak kaydedilmesi

### Ders planındaki yeri
**Ders 4:** Model güvenilirliği, yorumlanabilirlik ve raporlama.

## Workshop dosya yapısı

Bu seri çalıştırıldığında Google Drive altında aşağıdaki yapı oluşur:

```text
MyDrive/
└── CHEMBL206_workshop/
    ├── 01_data/
    ├── 02_cleaned/
    ├── 03_features/
    ├── 04_models/
    └── 05_reports/
```

> **Çalıştırma sırası:** Notebook 01 → 02 → 03 → 04 → 05  
> Her kod hücresinin sonunda bir **CHECKPOINT** mesajı vardır. Bir hücre hata verirse sonraki hücreye geçmeyiniz.

## Bilimsel not: Neden PCA-leverage?

Klasik leverage hesabı doğrudan tüm descriptorlarla yapılabilir. Ancak descriptor sayısı örnek sayısına yaklaştığında veya onu geçtiğinde hat matrisi kararsızlaşabilir ve kritik eşik anlamsızlaşabilir.

Bu notebook:

1. Featureları yalnızca **eğitim seti** üzerinde standardize eder.
2. PCA'yı yalnızca eğitim setinde fit eder.
3. Önce eğitim varyansının %95'ini açıklamak için gereken bileşen sayısını bulur.
4. `h*` eşiğinin anlamsız biçimde 1'in üzerine çıkmasını önlemek için bileşen sayısını eğitim örnek sayısına göre sınırlar.
5. Seçilen PCA skorlarında leverage hesaplar.

Kritik leverage eşiği:

\[
h^* = \frac{3(k+1)}{n_{train}}
\]

Burada `k` kullanılan PCA bileşeni sayısıdır.

Williams plot için artık eşiği `±3` standartlaştırılmış artık olarak alınır.

In [ ]:
# Google Drive bağlantısını kuruyoruz.
from google.colab import drive

# Drive'ı standart Colab yoluna bağlıyoruz.
drive.mount("/content/drive")

print("✅ CHECKPOINT 05.1: Google Drive bağlantısı kuruldu.")

In [ ]:
# SHAP ve analiz paketlerini Colab ortamına kuruyoruz.
!pip -q install shap scikit-learn joblib matplotlib

print("✅ CHECKPOINT 05.2: SHAP ve analiz paketleri kuruldu.")

In [ ]:
# Dosya yolları için pathlib kullanıyoruz.
from pathlib import Path

# Tablo işlemleri için pandas kullanıyoruz.
import pandas as pd

# Matris hesaplamaları için numpy kullanıyoruz.
import numpy as np

# JSON dosyalarını okumak için json kullanıyoruz.
import json

# Eğitilmiş modeli yüklemek için joblib kullanıyoruz.
import joblib

# Grafik çizimi için matplotlib kullanıyoruz.
import matplotlib.pyplot as plt

# SHAP açıklamaları için shap paketini içe aktarıyoruz.
import shap

# Feature standardizasyonu için StandardScaler kullanıyoruz.
from sklearn.preprocessing import StandardScaler

# PCA indirgemesi için PCA kullanıyoruz.
from sklearn.decomposition import PCA

# Notebook içinde tablo göstermek için display kullanıyoruz.
from IPython.display import display

# Proje klasörlerini tanımlıyoruz.
PROJECT_ROOT = Path("/content/drive/MyDrive/CHEMBL206_workshop")
MODEL_DIR = PROJECT_ROOT / "04_models"
REPORT_DIR = PROJECT_ROOT / "05_reports"

# Rapor klasörünü oluşturuyoruz.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Notebook 04 çıktılarının yollarını tanımlıyoruz.
MODEL_PATH = MODEL_DIR / "CHEMBL206_final_tree_model.joblib"
FEATURE_NAMES_PATH = MODEL_DIR / "CHEMBL206_final_feature_names.json"
TRAIN_PATH = MODEL_DIR / "CHEMBL206_train_with_predictions.csv"
TEST_PATH = MODEL_DIR / "CHEMBL206_test_with_predictions.csv"
MODEL_METADATA_PATH = MODEL_DIR / "CHEMBL206_final_model_metadata.json"

# Zorunlu dosyaların tamamını kontrol ediyoruz.
required_paths = [
    MODEL_PATH,
    FEATURE_NAMES_PATH,
    TRAIN_PATH,
    TEST_PATH,
    MODEL_METADATA_PATH,
]

# Eksik dosyaları listeliyoruz.
missing_paths = [path for path in required_paths if not path.exists()]

# Eksik çıktı varsa işlemi durduruyoruz.
if missing_paths:
    raise FileNotFoundError(
        "Notebook 04 çıktılarından bazıları bulunamadı:\n"
        + "\n".join(str(path) for path in missing_paths)
    )

print("Model dosyası:", MODEL_PATH)
print("Rapor klasörü:", REPORT_DIR)
print("✅ CHECKPOINT 05.3: Application domain ve SHAP girdileri bulundu.")

In [ ]:
# Eğitilmiş final modeli yüklüyoruz.
final_model = joblib.load(MODEL_PATH)

# Feature adlarını JSON dosyasından okuyoruz.
with open(FEATURE_NAMES_PATH, "r", encoding="utf-8") as file:
    feature_columns = json.load(file)

# Model metadata bilgisini okuyoruz.
with open(MODEL_METADATA_PATH, "r", encoding="utf-8") as file:
    model_metadata = json.load(file)

# Train ve test tablolarını okuyoruz.
train_df = pd.read_csv(TRAIN_PATH, low_memory=False)
test_df = pd.read_csv(TEST_PATH, low_memory=False)

# Feature matrislerini aynı sütun sırasıyla oluşturuyoruz.
X_train = train_df[feature_columns].astype(float)
X_test = test_df[feature_columns].astype(float)

# Gerçek ve tahmin değerlerini alıyoruz.
y_train = train_df["actual_pIC50"].astype(float)
y_test = test_df["actual_pIC50"].astype(float)
pred_train = train_df["predicted_pIC50"].astype(float)
pred_test = test_df["predicted_pIC50"].astype(float)

print("Final model:", model_metadata["selected_model_name"])
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

# Feature sırası ve matris boyutlarını doğruluyoruz.
assert X_train.shape[1] == len(feature_columns)
assert X_test.shape[1] == len(feature_columns)

print("✅ CHECKPOINT 05.4: Model, feature listesi ve train/test verileri yüklendi.")

In [ ]:
# StandardScaler'ı yalnızca eğitim verisi üzerinde fit ediyoruz.
scaler = StandardScaler()

# Eğitim featurelarını standardize ediyoruz.
X_train_scaled = scaler.fit_transform(X_train)

# Test featurelarına eğitimde öğrenilen dönüşümü uyguluyoruz.
X_test_scaled = scaler.transform(X_test)

# Önce tam PCA fit ederek kümülatif açıklanan varyansı hesaplıyoruz.
pca_full = PCA(
    svd_solver="full",
)

# Tam PCA yalnızca eğitim featurelarında fit edilir.
pca_full.fit(X_train_scaled)

# Kümülatif açıklanan varyansı hesaplıyoruz.
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

# Eğitim varyansının en az %95'ini açıklayan ilk bileşen sayısını buluyoruz.
k_for_95_percent = int(
    np.searchsorted(cumulative_variance, 0.95) + 1
)

# h* = 3(k+1)/n_train değerinin 1'in altında kalmasına yardım edecek üst sınırı hesaplıyoruz.
max_k_for_meaningful_hstar = max(
    1,
    int(np.floor(len(X_train) / 3)) - 1,
)

# PCA bileşen sayısını hem varyans hedefi hem de AD kararlılığına göre seçiyoruz.
selected_pca_components = min(
    k_for_95_percent,
    max_k_for_meaningful_hstar,
    X_train_scaled.shape[1],
    len(X_train) - 1,
)

# Seçilen bileşen sayısıyla final PCA nesnesini oluşturuyoruz.
pca = PCA(
    n_components=selected_pca_components,
    svd_solver="full",
)

# PCA'yı yalnızca eğitim featurelarında fit ediyoruz.
Z_train = pca.fit_transform(X_train_scaled)

# Test featurelarını aynı PCA uzayına yansıtıyoruz.
Z_test = pca.transform(X_test_scaled)

print("Orijinal feature sayısı:", X_train.shape[1])
print("%95 varyans için gereken bileşen:", k_for_95_percent)
print("h* kararlılık üst sınırı:", max_k_for_meaningful_hstar)
print("Seçilen PCA bileşeni:", Z_train.shape[1])
print("Gerçekte açıklanan toplam varyans:", float(pca.explained_variance_ratio_.sum()))

assert Z_train.shape[1] > 0
assert Z_train.shape[1] <= min(X_train.shape[1], len(X_train) - 1)

print("✅ CHECKPOINT 05.5: Eğitim tabanlı standardizasyon ve PCA tamamlandı.")

In [ ]:
# PCA skorlarına sabit terim/intercept sütunu ekleyen fonksiyonu tanımlıyoruz.
def add_intercept(matrix):
    # Solda 1'lerden oluşan sütun, sağda PCA skorları olacak biçimde birleştiriyoruz.
    return np.column_stack(
        [
            np.ones(matrix.shape[0]),
            matrix,
        ]
    )


# Eğitim ve test tasarım matrislerini oluşturuyoruz.
A_train = add_intercept(Z_train)
A_test = add_intercept(Z_test)

# (X'X)^-1 hesabında tekillik riskine karşı Moore–Penrose pseudo-inverse kullanıyoruz.
xtx_inverse = np.linalg.pinv(A_train.T @ A_train)

# Eğitim leverage değerlerini hat matrisinin köşegeninden hesaplıyoruz.
train_leverage = np.einsum(
    "ij,jk,ik->i",
    A_train,
    xtx_inverse,
    A_train,
)

# Test leverage değerlerini eğitim uzayına göre hesaplıyoruz.
test_leverage = np.einsum(
    "ij,jk,ik->i",
    A_test,
    xtx_inverse,
    A_test,
)

# Kritik leverage eşiğini hesaplıyoruz.
n_train = A_train.shape[0]
k_components = Z_train.shape[1]
h_star = 3 * (k_components + 1) / n_train

print("n_train:", n_train)
print("PCA bileşeni k:", k_components)
print("Kritik leverage h*:", h_star)
print("Train leverage aralığı:", (train_leverage.min(), train_leverage.max()))
print("Test leverage aralığı:", (test_leverage.min(), test_leverage.max()))

assert np.isfinite(train_leverage).all()
assert np.isfinite(test_leverage).all()

print("✅ CHECKPOINT 05.6: Train ve test leverage değerleri hesaplandı.")

In [ ]:
# Eğitim artıklarını hesaplıyoruz.
train_residuals = y_train.to_numpy() - pred_train.to_numpy()

# Test artıklarını hesaplıyoruz.
test_residuals = y_test.to_numpy() - pred_test.to_numpy()

# Artık ölçeğini yalnızca eğitim hatalarından tahmin ediyoruz.
# ddof=1 örnek standart sapmasını kullanır.
residual_scale = np.std(train_residuals, ddof=1)

# Sıfıra çok yakın ölçek durumunda sayısal hatayı engelliyoruz.
if residual_scale <= 1e-12:
    raise ValueError("Eğitim artık standart sapması sıfıra çok yakın.")

# Standartlaştırılmış eğitim artıklarını hesaplıyoruz.
train_std_residuals = train_residuals / residual_scale

# Standartlaştırılmış test artıklarını hesaplıyoruz.
test_std_residuals = test_residuals / residual_scale

# AD-in tanımını leverage ve ±3 artık koşullarının birlikte sağlanması olarak kullanıyoruz.
train_ad_in = (
    (train_leverage <= h_star)
    & (np.abs(train_std_residuals) <= 3)
)

# Test AD-in bayraklarını oluşturuyoruz.
test_ad_in = (
    (test_leverage <= h_star)
    & (np.abs(test_std_residuals) <= 3)
)

print("Train AD-in:", int(train_ad_in.sum()), "/", len(train_ad_in))
print("Train AD-out:", int((~train_ad_in).sum()))
print("Test AD-in:", int(test_ad_in.sum()), "/", len(test_ad_in))
print("Test AD-out:", int((~test_ad_in).sum()))

assert len(train_ad_in) == len(train_df)
assert len(test_ad_in) == len(test_df)

print("✅ CHECKPOINT 05.7: Standartlaştırılmış artıklar ve AD bayrakları üretildi.")

In [ ]:
# Train tablosuna application domain sütunlarını ekliyoruz.
train_ad_output = train_df.copy()
train_ad_output["leverage"] = train_leverage
train_ad_output["standardized_residual"] = train_std_residuals
train_ad_output["h_star"] = h_star
train_ad_output["AD_status"] = np.where(train_ad_in, "AD-in", "AD-out")

# Test tablosuna application domain sütunlarını ekliyoruz.
test_ad_output = test_df.copy()
test_ad_output["leverage"] = test_leverage
test_ad_output["standardized_residual"] = test_std_residuals
test_ad_output["h_star"] = h_star
test_ad_output["AD_status"] = np.where(test_ad_in, "AD-in", "AD-out")

# AD çıktı yollarını tanımlıyoruz.
TRAIN_AD_OUTPUT = REPORT_DIR / "CHEMBL206_train_application_domain.csv"
TEST_AD_OUTPUT = REPORT_DIR / "CHEMBL206_test_application_domain.csv"

# AD tablolarını kaydediyoruz.
train_ad_output.to_csv(TRAIN_AD_OUTPUT, index=False)
test_ad_output.to_csv(TEST_AD_OUTPUT, index=False)

# Williams plot çiziyoruz.
plt.figure(figsize=(10, 7))

# Eğitim noktalarını çiziyoruz.
plt.scatter(
    train_leverage,
    train_std_residuals,
    alpha=0.55,
    label="Train",
)

# Test noktalarını çiziyoruz.
plt.scatter(
    test_leverage,
    test_std_residuals,
    alpha=0.80,
    marker="^",
    label="Test",
)

# Kritik leverage çizgisini ekliyoruz.
plt.axvline(h_star, linestyle="--", label=f"h* = {h_star:.3f}")

# Standartlaştırılmış artık sınırlarını ekliyoruz.
plt.axhline(3, linestyle="--")
plt.axhline(-3, linestyle="--")

# Grafik etiketlerini ekliyoruz.
plt.xlabel("Leverage")
plt.ylabel("Standartlaştırılmış artık")
plt.title("CHEMBL206 — PCA tabanlı Williams Plot")
plt.legend()
plt.tight_layout()

# Williams plot'u kaydediyoruz.
WILLIAMS_PLOT = REPORT_DIR / "CHEMBL206_Williams_plot_PCA_leverage.png"
plt.savefig(WILLIAMS_PLOT, dpi=300, bbox_inches="tight")
plt.show()

print("Train AD tablosu:", TRAIN_AD_OUTPUT)
print("Test AD tablosu:", TEST_AD_OUTPUT)
print("Williams plot:", WILLIAMS_PLOT)

assert WILLIAMS_PLOT.exists()

print("✅ CHECKPOINT 05.8: Application domain tabloları ve Williams plot kaydedildi.")

In [ ]:
# SHAP hesabının workshop süresini kontrol etmek için maksimum örnek sayısını tanımlıyoruz.
MAX_SHAP_SAMPLES = 200

# Test setinden açıklanacak örnekleri seçiyoruz.
# Test seti 200'den küçükse tamamı kullanılır.
X_shap = X_test.sample(
    n=min(MAX_SHAP_SAMPLES, len(X_test)),
    random_state=42,
)

# TreeExplainer nesnesini final ağaç modeli için oluşturuyoruz.
explainer = shap.TreeExplainer(final_model)

# Yeni SHAP Explanation API ile açıklamaları hesaplıyoruz.
shap_values = explainer(X_shap)

print("SHAP örnek sayısı:", X_shap.shape[0])
print("SHAP feature sayısı:", X_shap.shape[1])
print("SHAP değer matrisi:", shap_values.values.shape)

assert shap_values.values.shape == X_shap.shape

print("✅ CHECKPOINT 05.9: SHAP değerleri hesaplandı.")

In [ ]:
# Global önem için her featureın ortalama mutlak SHAP değerini hesaplıyoruz.
mean_abs_shap = np.abs(shap_values.values).mean(axis=0)

# Feature önem tablosunu oluşturuyoruz.
shap_importance = pd.DataFrame(
    {
        "feature": feature_columns,
        "mean_abs_SHAP": mean_abs_shap,
    }
).sort_values(
    "mean_abs_SHAP",
    ascending=False,
)

# En önemli 20 descriptorı gösteriyoruz.
display(shap_importance.head(20))

# Önem tablosunu CSV olarak kaydediyoruz.
SHAP_IMPORTANCE_OUTPUT = REPORT_DIR / "CHEMBL206_SHAP_feature_importance.csv"
shap_importance.to_csv(SHAP_IMPORTANCE_OUTPUT, index=False)

print("SHAP önem tablosu:", SHAP_IMPORTANCE_OUTPUT)
print("✅ CHECKPOINT 05.10: Global SHAP önem sıralaması oluşturuldu.")

In [ ]:
# SHAP beeswarm grafiğini çiziyoruz.
shap.plots.beeswarm(
    shap_values,
    max_display=20,
    show=False,
)

# Başlık ve yerleşim ayarlarını ekliyoruz.
plt.title("CHEMBL206 — SHAP Beeswarm (Test örnekleri)")
plt.tight_layout()

# Beeswarm grafiğini kaydediyoruz.
SHAP_BEESWARM_OUTPUT = REPORT_DIR / "CHEMBL206_SHAP_beeswarm.png"
plt.savefig(SHAP_BEESWARM_OUTPUT, dpi=300, bbox_inches="tight")
plt.show()

# SHAP bar grafiğini çiziyoruz.
shap.plots.bar(
    shap_values,
    max_display=20,
    show=False,
)

# Başlık ve yerleşim ayarlarını ekliyoruz.
plt.title("CHEMBL206 — Ortalama |SHAP| Feature Önemi")
plt.tight_layout()

# Bar grafiğini kaydediyoruz.
SHAP_BAR_OUTPUT = REPORT_DIR / "CHEMBL206_SHAP_bar.png"
plt.savefig(SHAP_BAR_OUTPUT, dpi=300, bbox_inches="tight")
plt.show()

print("SHAP beeswarm:", SHAP_BEESWARM_OUTPUT)
print("SHAP bar:", SHAP_BAR_OUTPUT)

assert SHAP_BEESWARM_OUTPUT.exists()
assert SHAP_BAR_OUTPUT.exists()

print("✅ CHECKPOINT 05.11: Global SHAP grafikleri kaydedildi.")

In [ ]:
# Lokal açıklama için test setinde mutlak hatası en büyük örneği seçiyoruz.
largest_error_position = int(
    np.argmax(
        np.abs(
            test_df["actual_pIC50"].to_numpy()
            - test_df["predicted_pIC50"].to_numpy()
        )
    )
)

# Seçilen örneğin feature satırını alıyoruz.
X_local = X_test.iloc[[largest_error_position]]

# Aynı örnek için SHAP açıklamasını hesaplıyoruz.
local_shap = explainer(X_local)

# Waterfall grafiğini çiziyoruz.
shap.plots.waterfall(
    local_shap[0],
    max_display=15,
    show=False,
)

# Başlık ve yerleşim ayarlarını ekliyoruz.
plt.title("CHEMBL206 — En büyük test hatası için lokal SHAP")
plt.tight_layout()

# Lokal SHAP grafiğini kaydediyoruz.
SHAP_LOCAL_OUTPUT = REPORT_DIR / "CHEMBL206_SHAP_local_waterfall.png"
plt.savefig(SHAP_LOCAL_OUTPUT, dpi=300, bbox_inches="tight")
plt.show()

# Seçilen molekülün özetini yazdırıyoruz.
local_info_columns = [
    column
    for column in [
        "molecule_id",
        "parent_smiles",
        "actual_pIC50",
        "predicted_pIC50",
        "residual",
    ]
    if column in test_df.columns
]

display(test_df.iloc[[largest_error_position]][local_info_columns])

print("Lokal SHAP grafiği:", SHAP_LOCAL_OUTPUT)
print("✅ CHECKPOINT 05.12: Lokal SHAP açıklaması oluşturuldu.")

In [ ]:
# En önemli descriptor adını seçiyoruz.
top_feature = shap_importance.iloc[0]["feature"]

# En önemli descriptor için SHAP saçılım grafiğini çiziyoruz.
shap.plots.scatter(
    shap_values[:, top_feature],
    show=False,
)

# Başlık ve yerleşim ayarlarını ekliyoruz.
plt.title(f"CHEMBL206 — SHAP Dependence: {top_feature}")
plt.tight_layout()

# Dependence grafiğini kaydediyoruz.
SHAP_DEPENDENCE_OUTPUT = REPORT_DIR / "CHEMBL206_SHAP_top_feature_dependence.png"
plt.savefig(SHAP_DEPENDENCE_OUTPUT, dpi=300, bbox_inches="tight")
plt.show()

print("En önemli descriptor:", top_feature)
print("Dependence grafiği:", SHAP_DEPENDENCE_OUTPUT)
print("✅ CHECKPOINT 05.13: En önemli descriptor için SHAP dependence grafiği oluşturuldu.")

In [ ]:
# Workshop final rapor özetini hazırlıyoruz.
final_summary = pd.DataFrame(
    {
        "metric": [
            "final_model",
            "n_train",
            "n_test",
            "n_original_features",
            "n_PCA_components_for_AD",
            "PCA_explained_variance",
            "h_star",
            "train_AD_in",
            "train_AD_out",
            "test_AD_in",
            "test_AD_out",
            "top_SHAP_feature",
        ],
        "value": [
            model_metadata["selected_model_name"],
            len(train_df),
            len(test_df),
            len(feature_columns),
            Z_train.shape[1],
            pca.explained_variance_ratio_.sum(),
            h_star,
            int(train_ad_in.sum()),
            int((~train_ad_in).sum()),
            int(test_ad_in.sum()),
            int((~test_ad_in).sum()),
            top_feature,
        ],
    }
)

# Final özeti ekranda gösteriyoruz.
display(final_summary)

# Final özeti CSV olarak kaydediyoruz.
FINAL_SUMMARY_OUTPUT = REPORT_DIR / "CHEMBL206_workshop_final_summary.csv"
final_summary.to_csv(FINAL_SUMMARY_OUTPUT, index=False)

print("Final workshop özeti:", FINAL_SUMMARY_OUTPUT)
print("✅ CHECKPOINT 05.14: Tüm pipeline başarıyla tamamlandı.")

## Workshop kapanış tartışması

1. Bir molekülün AD-out olması tahminin kesinlikle yanlış olduğu anlamına gelir mi?
2. Leverage yüksek, standartlaştırılmış artık düşükse molekülü nasıl yorumlarız?
3. SHAP nedensellik gösterir mi, yoksa model davranışını mı açıklar?
4. Mordred descriptorları arasında korelasyon varken SHAP önemleri nasıl etkilenebilir?
5. Sonraki geliştirme adımı olarak random split yerine scaffold split nasıl eklenebilir?

## Üretilen ana çıktılar

- LazyPredict karşılaştırma tablosu
- Final ağaç modeli (`.joblib`)
- Train/test gerçek ve tahmin pIC50 tabloları
- 5-fold CV özeti
- PCA-leverage Williams plot
- AD-in / AD-out CSV dosyaları
- SHAP beeswarm, bar, waterfall ve dependence grafikleri
- SHAP feature importance CSV dosyası